In [1]:
import os
# 💡 PROTEÇÃO: Limpa o conflito de baixo nível do Windows antes de qualquer import
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import glob
import pickle
import time as tm
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from scipy.signal import find_peaks
import ross as rs

# ==============================================================================
# DATASET PARAMÉTRICO LEVE (SEM MATRIZES DENSAS NA MEMÓRIA)
# ==============================================================================
class FastRotorStateDataset(Dataset):
    def __init__(self, pasta_lotes):
        super().__init__()
        self.samples = []
        
        if isinstance(pasta_lotes, str):
            padrao_busca = os.path.join(pasta_lotes, "lote_*.pkl")
            arquivos_encontrados = glob.glob(padrao_busca)
            print(f"📂 Carregando lotes do disco para o Dataset...")
            for caminho_arquivo in arquivos_encontrados:
                with open(caminho_arquivo, "rb") as f:
                    dados_lote = pickle.load(f)
                if isinstance(dados_lote, list):
                    self.samples += dados_lote
            print(f"✅ Sucesso: {len(self.samples)} amostras prontas.")
        else:
            self.samples = pasta_lotes

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        chaves_kxx = sorted([k for k in sample.keys() if k.startswith('kxx_')])
        num_mancais = len(chaves_kxx) if len(chaves_kxx) > 0 else 2
        
        mancais_features = []
        for idx_b in range(num_mancais):
            kxx = np.log10(sample.get(f'kxx_{idx_b}', sample.get('kxx', 1e6)) + 1e-5)
            kyy = np.log10(sample.get(f'kyy_{idx_b}', sample.get('kyy', 1.2e6)) + 1e-5)
            cxx = np.log10(sample.get(f'cxx_{idx_b}', sample.get('cxx', 1e3)) + 1e-5)
            cyy = np.log10(sample.get(f'cyy_{idx_b}', sample.get('cyy', 1.1e3)) + 1e-5)
            mancais_features.extend([kxx, kyy, cxx, cyy])
            
        gdl_in = float(sample['Node_in'])
        gdl_out = float(sample['Node_out'])
        x_design = np.concatenate([mancais_features, [gdl_in, gdl_out]])
        
        if 'FRF' in sample:
            frf_log = np.log10(np.abs(sample['FRF']) + 1e-15)
            y_tensor = torch.tensor(frf_log, dtype=torch.float)
        else:
            y_tensor = torch.zeros(1000, dtype=torch.float)
        
        return {'x': torch.tensor(x_design, dtype=torch.float), 'y': y_tensor}

# ==============================================================================
# REDE NEURAL FOURIER + MLP
# ==============================================================================
class PureFourierFeatureLayer(nn.Module):
    def __init__(self, input_dim, mapping_size=256, scale=10.0):
        super().__init__()
        B = torch.randn(input_dim, mapping_size) * scale
        self.register_buffer('B', B)

    def forward(self, x):
        projection = 2 * np.pi * torch.matmul(x, self.B)
        return torch.cat([torch.sin(projection), torch.cos(projection)], dim=-1)

class FastRotorFourierSurrogate(nn.Module):
    def __init__(self, input_dim, mapping_size=256, num_frequencies=1000):
        super().__init__()
        self.fourier = PureFourierFeatureLayer(input_dim=input_dim, mapping_size=mapping_size, scale=10.0)
        fourier_dim = mapping_size * 2
        
        self.mlp = nn.Sequential(
            nn.Linear(fourier_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, num_frequencies)
        )
        
    def forward(self, x):
        return self.mlp(self.fourier(x))

# ==============================================================================
# PIPELINE GERENCIADOR DA IA
# ==============================================================================
class RotorSurrogatePipeline:
    def __init__(self, mapping_size=256, epochs=150, batch_size=64, lr=0.001):
        self.mapping_size = mapping_size
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None

    def train_pipeline(self, pasta_lotes):
        train_dataset = FastRotorStateDataset(pasta_lotes)
        train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)
        
        input_dim = train_dataset[0]['x'].shape[0]
        self.model = FastRotorFourierSurrogate(input_dim=input_dim, mapping_size=self.mapping_size).to(self.device)
        
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        criterion = nn.MSELoss()
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        
        print(f"\n🚀 Treinando a rede paramétrica na ({self.device})...")
        for epoch in range(1, self.epochs + 1):
            self.model.train()
            total_loss = 0
            for batch in train_loader:
                x_batch = batch['x'].to(self.device)
                y_batch = batch['y'].to(self.device)
                
                optimizer.zero_grad()
                out = self.model(x_batch)
                loss = criterion(out, y_batch)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * x_batch.size(0)
                
            epoch_loss = total_loss / len(train_dataset)
            scheduler.step(epoch_loss)
            if epoch % 25 == 0 or epoch == 1:
                print(f"   Época {epoch:03d}/{self.epochs} | Loss MSE: {epoch_loss:.6f}")
        print("✅ Treinamento Concluído com Sucesso!")

    def save_model(self, caminho_no_disco):
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'input_dim': self.model.fourier.B.shape[0],
            'mapping_size': self.mapping_size
        }, caminho_no_disco)
        print(f"💾 Modelo salvo em: {caminho_no_disco}")

    def load_model(self, caminho_no_disco):
        checkpoint = torch.load(caminho_no_disco, map_location=self.device)
        self.mapping_size = checkpoint['mapping_size']
        self.model = FastRotorFourierSurrogate(input_dim=checkpoint['input_dim'], mapping_size=self.mapping_size).to(self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        print(f"🔌 IA Carregada e Pronta!")

    def predict_frf_batch(self, samples_list):
        self.model.eval()
        eval_dataset = FastRotorStateDataset(samples_list)
        eval_loader = DataLoader(eval_dataset, batch_size=len(eval_dataset), shuffle=False)
        batch = next(iter(eval_loader))
        x_tensor = batch['x'].to(self.device)
        with torch.no_grad():
            y_pred_log = self.model(x_tensor).cpu().numpy()
        return 10**y_pred_log

c:\Users\Cliente\anaconda3\envs\ROSS_IA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Rotor_Trust:
    def __init__(self, rotor, params, speed=None):
        self.rotor = rotor
        self.params = params
        self.bearing_base = copy.deepcopy(rotor.bearing_elements)
        self.speed = np.linspace(0, 1000, 1000) if speed is None else speed
        self.tempo_ross_total = 0.0 # Guarda o tempo que o ROSS levou para gerar as amostras

    def units_speed(self, speed_to_convert):
        return np.array(speed_to_convert) * 60 / (2 * np.pi)

    def _get_auto_operating_zone(self, frf_data):
        res_items = self.items_analysis(frf_data)
        if res_items is None or len(res_items[2]) < 2: return None, None
        peaks = np.array(sorted(res_items[2]))
        gaps = np.diff(peaks)
        idx = np.argmax(gaps)
        return 1.1 * peaks[idx], 0.9 * peaks[idx + 1]

    def _sma_internal(self, nma, nmc, nc):
        nc = np.sort(np.atleast_1d(nc))
        if nc.size == 0: return [0.0]
        below = nc[nc < nma]
        above = nc[nc > nmc]
        if len(below) > 0: return [100 * ((nma - below[-1]) / nma)]
        elif len(above) > 0: return [100 * ((above[0] - nmc) / nmc)]
        return [0.0]

    def _smr_internal(self, Q, nc, nma, nmc):
        nc = np.sort(np.atleast_1d(nc))
        Q = np.atleast_1d(Q)
        if nc.size == 0 or np.max(Q) < 2.5: return [0.0]
        below = nc[nc < nma]
        above = nc[nc > nmc]
        if len(below) > 0: return [17 * (1 - (1 / (Q[-1] - 1.5)))]
        elif len(above) > 0: return [10 + (17 * (1 - (1 / (Q[0] - 1.5))))]
        return [0.0]

    def _extract_modal_internal(self, frf_vals):
        peaks_idx, _ = find_peaks(frf_vals)
        if len(peaks_idx) == 0: return None
        pk_index = peaks_idx[np.argmax(frf_vals[peaks_idx])]
        h_target = 0.707 * frf_vals[pk_index]
        nc_rpm = self.units_speed(self.speed[pk_index])
        try:
            xn1 = np.interp(h_target, frf_vals[:pk_index], self.speed[:pk_index])
            xn2 = np.interp(h_target, frf_vals[pk_index:][::-1], self.speed[pk_index:][::-1])
            delta_n = self.units_speed(xn2) - self.units_speed(xn1)
            if delta_n <= 0: return None
            return nc_rpm, (nc_rpm / delta_n)
        except: return None

    def items_analysis(self, frf_data):
        peaks, _ = find_peaks(frf_data)
        if len(peaks) == 0: return None
        h_peaks = frf_data[peaks]
        h_target = 0.707 * h_peaks
        n1_end, n2_end = [], []
        for i, pk in enumerate(peaks):
            idx_l, idx_r = pk, pk
            while idx_l > 0 and frf_data[idx_l] > h_target[i]: idx_l -= 1
            while idx_r < len(frf_data)-1 and frf_data[idx_r] > h_target[i]: idx_r += 1
            n1_end.append(np.interp(h_target[i], frf_data[idx_l:idx_l+2], self.speed[idx_l:idx_l+2]))
            n2_end.append(np.interp(h_target[i], frf_data[idx_r-1:idx_r+1][::-1], self.speed[idx_r-1:idx_r+1][::-1]))
        return (h_peaks, h_target, self.units_speed(self.speed[peaks]), self.units_speed(n1_end), self.units_speed(n2_end))

    # ==============================================================================
    # GERAÇÃO UNIFORME DE ALTA AMPLITUDE (+- 70%) EM LOTES
    # ==============================================================================
    def generate_and_save_in_batches(self, N_samples, pasta_destino, tamanho_lote=500):
        import gc
        os.makedirs(pasta_destino, exist_ok=True)
        for f in glob.glob(os.path.join(pasta_destino, "lote_*.pkl")): os.remove(f)
            
        print(f"📂 Gerando {N_samples} amostras ASSIMÉTRICAS via Distribuição Uniforme (±70%)...")
        t_inicial = tm.time()
        lote_atual, contador_lotes = [], 0
        n_nos = len(self.rotor.nodes)
        num_mancais = len(self.bearing_base)
        
        for sim in range(N_samples):
            sampled_values = {}
            for coef in ['kxx', 'kyy', 'cxx', 'cyy']:
                if coef in self.params:
                    data = self.params[coef]
                    for idx_b in range(num_mancais):
                        # 💡 ALTERADO AQUI: Intervalo aberto para +- 70% conforme pedido
                        sampled_values[f'{coef}_{idx_b}'] = float(np.random.uniform(data['mi']*0.3, data['mi']*1.7))
            
            shaft = copy.deepcopy(self.rotor.shaft_elements)
            disks = copy.deepcopy(self.rotor.disk_elements)
            seals = copy.deepcopy(getattr(self.rotor, 'seal_elements', []))
            new_bearings = []
            for i, b in enumerate(self.bearing_base):
                new_bearings.append(rs.BearingElement(n=b.n, 
                                                      kxx=sampled_values[f'kxx_{i}'], kyy=sampled_values[f'kyy_{i}'], 
                                                      cxx=sampled_values[f'cxx_{i}'], cyy=sampled_values[f'cyy_{i}']))
            
            new_rotor = rs.Rotor(shaft, disks, new_bearings + seals)
            results = new_rotor.run_freq_response(speed_range=self.speed)
            
            direction = np.random.randint(0, 2)
            fin = np.random.randint(0, n_nos) * 6 + direction
            fout = np.random.randint(0, n_nos) * 6 + direction
            
            row = {
                'Node_in': fin, 'Node_out': fout, 'FRF': np.abs(results.freq_resp[fout, fin, :]),
                **sampled_values
            }
            lote_atual.append(row)
            
            if len(lote_atual) == tamanho_lote or sim == N_samples - 1:
                contador_lotes += 1
                with open(os.path.join(pasta_destino, f"lote_{contador_lotes}.pkl"), "wb") as f:
                    pickle.dump(lote_atual, f)
                del lote_atual
                lote_atual = []
                gc.collect() # Libera a RAM dinamicamente para o kernel não sufocar
                
        self.tempo_ross_total = tm.time() - t_inicial
        print(f"✅ Geração uniforme concluída em {self.tempo_ross_total:.2f}s!")

    # ==============================================================================
    # 📊 VALIDAÇÃO E RELATÓRIO DE ERROS / TEMPOS LOGO NO INÍCIO
    # ==============================================================================
    def run_mc_error_validation(self, pipeline_ia, pasta_lotes, lv=None, nma=None, nmc=None):
        print(f"\n📊 Iniciando Validação Cruzada Direta (ROSS vs IA)...")
        dataset_validacao = FastRotorStateDataset(pasta_lotes)
        N_samples = len(dataset_validacao)
        lista_amostras = dataset_validacao.samples
        
        todas_frfs_ross = np.array([sample['FRF'] for sample in lista_amostras])
        
        # Medição do tempo de processamento da IA
        t_ini_ia = tm.time()
        todas_frfs_ia = pipeline_ia.predict_frf_batch(lista_amostras)
        tempo_ia_total = tm.time() - t_ini_ia
        
        if lv is None: lv = np.max(todas_frfs_ross[0]) * 1.1
        if nma is None or nmc is None: nma, nmc = self._get_auto_operating_zone(todas_frfs_ross[0])
            
        h_lv_ross, h_sm_ross = np.zeros(N_samples), np.zeros(N_samples)
        h_lv_ia, h_sm_ia = np.zeros(N_samples), np.zeros(N_samples)
        
        for i in range(N_samples):
            frf_ross, frf_ia = todas_frfs_ross[i, :], todas_frfs_ia[i, :]
            if np.max(frf_ross) > lv: h_lv_ross[i] = 1
            if np.max(frf_ia) > lv: h_lv_ia[i] = 1
                
            modal_ross = self._extract_modal_internal(frf_ross)
            if modal_ross is not None:
                nc, Q = modal_ross
                if (self._sma_internal(nma, nmc, nc)[0] - self._smr_internal(Q, nc, nma, nmc)[0] < 0): h_sm_ross[i] = 1
                    
            modal_ia = self._extract_modal_internal(frf_ia)
            if modal_ia is not None:
                nc, Q = modal_ia
                if (self._sma_internal(nma, nmc, nc)[0] - self._smr_internal(Q, nc, nma, nmc)[0] < 0): h_sm_ia[i] = 1

        pf_lv_ross_conv = np.cumsum(h_lv_ross) / np.arange(1, N_samples + 1)
        pf_lv_ia_conv = np.cumsum(h_lv_ia) / np.arange(1, N_samples + 1)
        pf_sm_ross_conv = np.cumsum(h_sm_ross) / np.arange(1, N_samples + 1)
        pf_sm_ia_conv = np.cumsum(h_sm_ia) / np.arange(1, N_samples + 1)
        
        # PLOTS DE COMPARAÇÃO
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        ax1.plot(range(1, N_samples + 1), pf_lv_ross_conv, label=f'ROSS Real ({pf_lv_ross_conv[-1]:.3e})', color='black', linewidth=2)
        ax1.plot(range(1, N_samples + 1), pf_lv_ia_conv, label=f'IA Predição ({pf_lv_ia_conv[-1]:.3e})', color='blue', linestyle='--')
        ax1.set_title("Convergência de Pf - Limite de Vibração"); ax1.set_xlabel("Amostras do Disco"); ax1.grid(True, alpha=0.3); ax1.legend()
        
        ax2.plot(range(1, N_samples + 1), pf_sm_ross_conv, label=f'ROSS Real ({pf_sm_ross_conv[-1]:.3e})', color='black', linewidth=2)
        ax2.plot(range(1, N_samples + 1), pf_sm_ia_conv, label=f'IA Predição ({pf_sm_ia_conv[-1]:.3e})', color='green', linestyle='--')
        ax2.set_title("Convergência de Pf - Margem API 617"); ax2.set_xlabel("Amostras do Disco"); ax2.grid(True, alpha=0.3); ax2.legend()
        
        # Exibição explícita do balanço de tempos no título global
        ganho_velocidade = self.tempo_ross_total / max(tempo_ia_total, 1e-3)
        plt.suptitle(f"Validação Cruzada (Amostras Uniformes: {N_samples})\n⏱️ Tempo ROSS: {self.tempo_ross_total:.2f}s | 🧠 Tempo IA: {tempo_ia_total:.2f}s [Ganho de {ganho_velocidade:.1f}x]", fontsize=12)
        plt.tight_layout()
        plt.show()
        
        return {"erro_pf_lv": abs(pf_lv_ross_conv[-1] - pf_lv_ia_conv[-1]), "erro_pf_sm": abs(pf_sm_ross_conv[-1] - pf_sm_ia_conv[-1])}

    # ==============================================================================
    # MONTE CARLO MONUMENTAL DE CONFIABILIDADE (IA PURA)
    # ==============================================================================
    def run_pure_surrogate_mc(self, pipeline_ia, target_params, N_simulations=3000, mancais_iguais=True, 
                              lv=None, nma=None, nmc=None, node_in=None, node_out=None):
        print(f"\n🧠 Executando Monte Carlo de Produção via IA-Only (N={N_simulations})...")
        t_ini = tm.time()
        num_mancais = len(self.bearing_base)
        n_nos = len(self.rotor.nodes)
        
        ref_in = node_in if node_in is not None else 1
        ref_out = node_out if node_out is not None else 5
        
        mock_nominal = {}
        for coef in ['kxx', 'kyy', 'cxx', 'cyy']:
            for idx_b in range(num_mancais): mock_nominal[f'{coef}_{idx_b}'] = target_params[coef]['mi']
        mock_nominal.update({'Node_in': float(ref_in), 'Node_out': float(ref_out)})
        
        frf_nom = pipeline_ia.predict_frf_batch([mock_nominal])[0]
        if nma is None or nmc is None: nma, nmc = self._get_auto_operating_zone(frf_nom)
        if lv is None: lv = np.max(frf_nom) * 1.1
            
        lista_amostras_input = []
        for i in range(N_simulations):
            row_amostra = {}
            for coef in ['kxx', 'kyy', 'cxx', 'cyy']:
                if coef in target_params:
                    data = target_params[coef]
                    if mancais_iguais:
                        val = float(np.random.normal(data['mi'], data['sigma']))
                        for idx_b in range(num_mancais): row_amostra[f'{coef}_{idx_b}'] = val
                    else:
                        for idx_b in range(num_mancais):
                            row_amostra[f'{coef}_{idx_b}'] = float(np.random.normal(data['mi'], data['sigma']))
            
            if node_in is not None and node_out is not None:
                row_amostra['Node_in'] = float(node_in)
                row_amostra['Node_out'] = float(node_out)
            else:
                direction = np.random.randint(0, 2)
                row_amostra['Node_in'] = float(np.random.randint(0, n_nos) * 6 + direction)
                row_amostra['Node_out'] = float(np.random.randint(0, n_nos) * 6 + direction)
            
            lista_amostras_input.append(row_amostra)

        todas_frfs_ia = pipeline_ia.predict_frf_batch(lista_amostras_input)
        
        h_lv, h_sm = np.zeros(N_simulations), np.zeros(N_simulations)
        for i in range(N_simulations):
            frf_ia = todas_frfs_ia[i, :]
            if np.max(frf_ia) > lv: h_lv[i] = 1
            modal = self._extract_modal_internal(frf_ia)
            if modal is not None:
                nc, Q = modal
                if (self._sma_internal(nma, nmc, nc)[0] - self._smr_internal(Q, nc, nma, nmc)[0] < 0): h_sm[i] = 1

        pf_lv_conv = np.cumsum(h_lv) / np.arange(1, N_simulations + 1)
        pf_sm_conv = np.cumsum(h_sm) / np.arange(1, N_simulations + 1)
        t_fim = tm.time() - t_ini
        
        plt.figure(figsize=(10, 4))
        plt.plot(range(1, N_simulations + 1), pf_lv_conv, label=f'Pf LV: {pf_lv_conv[-1]:.3e}', color='blue')
        plt.plot(range(1, N_simulations + 1), pf_sm_conv, label=f'Pf SM (API): {pf_sm_conv[-1]:.3e}', color='green')
        plt.title(f"Confiabilidade de Produção via IA Otimizada\n⏱️ Tempo de Execução IA: {t_fim:.2f}s | N={N_simulations}")
        plt.xlabel("Simulações Realocadas"); plt.ylabel("Probabilidade de Falha"); plt.grid(True, alpha=0.3); plt.legend()
        plt.show()
        
        return {"pf_lv": pf_lv_conv[-1], "pf_sm": pf_sm_conv[-1], "tempo_s": t_fim}

In [3]:

steel1 = rs.Material(name="Steel", rho=7810, E=2.11e11, Poisson=0.3)
shaft1 = [rs.ShaftElement(L=0.22, idl=0, odl=0.05, material=steel1,) for _ in range(7)]

disk11 = rs.DiskElement.from_geometry(n=3, material=steel1, width=0.07, i_d=0.05, o_d=0.28)
disk21 = rs.DiskElement.from_geometry(n=5, material=steel1, width=0.07, i_d=0.05, o_d=0.35)

bearing11 = rs.BearingElement(n=0, kxx=6.5e7, kyy=6.5e7, cxx=6.5e2, cyy=6.5e2)
bearing21 = rs.BearingElement(n=7, kxx=6.5e7, kyy=6.5e7, cxx=6.5e2, cyy=6.5e2)

disks1 = [disk11, disk21]
bearings1 = [bearing11, bearing21]
rotor1 = rs.Rotor(shaft1, disks1, bearings1)

rotor1.plot_rotor()

: 

In [ ]:
speed_range = np.linspace(0, 1000, 1000)

In [ ]:
# caminhos de controle do seu projeto
pasta_dados_lotes = r"C:\\Users\\Cliente\\Desktop\\Pastas\\Códigos\\surrogate_rotor_model\\dados_lotes"
caminho_pesos_ia = r"C:\\Users\\Cliente\\Desktop\\Pastas\\Códigos\\surrogate_rotor_model\\modelo_rotor_A\\modelo_rotor_A.pth"

params_nominais = {
    'kxx': {'mi': 1e6, 'sigma': 4e4},
    'kyy': {'mi': 1.2e6, 'sigma': 4e4},
    'cxx': {'mi': 1e3, 'sigma': 8e1},
    'cyy': {'mi': 1.1e3, 'sigma': 8e1}
}

# 1. Instancia a classe de confiabilidade com o seu objeto 'rotor' e parâmetros nominais
trust = Rotor_Trust(rotor=rotor1, params=params_nominais, speed=speed_range)

# 🚀 PASSO A: Geração Monumental de 10.000 dados (Uniforme ±70%)
# O tamanho do lote foi subido para 500 para acelerar a gravação no NVMe/SSD
trust.generate_and_save_in_batches(N_samples=10000, pasta_destino=pasta_dados_lotes, tamanho_lote=500)

# 🚀 PASSO B: Treinamento Robusto da Inteligência Neural
pipeline_ia = RotorSurrogatePipeline(epochs=150, batch_size=64) # Batch de 64 ajuda na estabilidade de 10k amostras
pipeline_ia.train_pipeline(pasta_dados_lotes)
pipeline_ia.save_model(caminho_pesos_ia)

# ==============================================================================
# 📊 PASSO C: VALIDAÇÃO CRUZADA IMEDIATA (O que você pediu!)
# ==============================================================================
erros_validacao = trust.run_mc_error_validation(
    pipeline_ia=pipeline_ia,
    pasta_lotes=pasta_dados_lotes
)
print("\n📈 Relatório Técnico de Discrepância Absoluta de Confiabilidade:")
print(f"   - Erro de Pf no Limite de Vibração: {erros_validacao['erro_pf_lv']:.5e}")
print(f"   - Erro de Pf na Margem da API 617: {erros_validacao['erro_pf_sm']:.5e}")

In [ ]:
# Modifica as dispersões para um novo estudo de caso
novos_params_estudo = {
    'kxx': {'mi': 1.0e6, 'sigma': 6e4},  # Aumentou a incerteza do mancal
    'kyy': {'mi': 1.2e6, 'sigma': 6e4},
    'cxx': {'mi': 1e3, 'sigma': 9e1},
    'cyy': {'mi': 1.1e3, 'sigma': 9e1}
}

# Roda 3.000 simulações de Monte Carlo acoplando os critérios de falha da API em segundos!
relatorio_ia_pura = trust.run_pure_surrogate_mc(
    pipeline_ia=pipeline_ia,
    target_params=novos_params_estudo,
    N_simulations=5000,
    mancais_iguais=True, # Alterne para False para avaliar o efeito de assimetria!
    lv=2e-05
)
print("Resultado Confiabilidade IA Pura:", relatorio_ia_pura)

In [ ]:
# Roda a comparação direta usando os dados estáticos salvos no disco
trust = Rotor_Trust(rotor=rotor1, params=params_estocasticos, speed=speed_range)

relatorio_validacao = trust.run_mc_error_validation(
    pipeline_ia=pipeline_ia,
    pasta_lotes=pasta_dados_lotes
)

In [ ]:
resultados_ia = trust.run_pure_surrogate_mc(
    pipeline_ia=pipeline_ia,
    target_params=params_estocasticos,
    N_simulations=3000,
    mancais_iguais=False,
    node_in=18,   # Excitação fixa no GDL 18
    node_out=0    # Medição fixa no GDL 0
)

In [ ]:
relatorio_ia_pura = trust.run_pure_surrogate_mc(
    pipeline_ia=pipeline_ia,
    target_params=params_estocasticos,
    N_simulations=5000,
    mancais_iguais=True, # Alterne para False para avaliar o efeito de assimetria!
    lv=2e-05,
    node_in = 18,
    node_out = 0
)